# einops-rearrange — worked example 3: Depth-to-space (pixel shuffle) via one rearrange

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Concept

A single `rearrange` pattern can decompose, reorder, and compose all at once. Depth-to-space upsampling takes `(b, c*r*r, h, w)` and produces `(b, c, h*r, w*r)`: the `r*r` extra channels become a small spatial block interleaved into each pixel location. This is the einops one-liner equivalent of `torch.nn.PixelShuffle`.

## Worked solution

We start with `(b, C, h, w)` where the channel count `C = c * r * r`. We want to spread those `r*r` factors out spatially, giving `(b, c, h*r, w*r)` — an `r×` upsample with no learned weights, just a relayout.

**Step 1 — decompose the packed channel axis.** Write the input channel axis as `(c r1 r2)`. We bind `r1=r` and `r2=r`; einops infers `c = C // (r*r)`. Now we have named sub-axes `r1` and `r2` that represent the within-pixel block.

**Step 2 — weave the block factors next to the spatial axes.** On the output side we write `h (h r1)`... no — carefully: we compose `(h r1)` to make the new height and `(w r2)` to make the new width. Inside `(h r1)`, `h` varies slowest and `r1` fastest, so each original row expands into `r` consecutive output rows.

**Step 3 — the full pattern.** `'b (c r1 r2) h w -> b c (h r1) (w r2)'` with `r1=r, r2=r`. einops handles the decompose (left parens), the reorder (moving `r1`/`r2` over to the spatial groups), and the compose (right parens) in one shot.

**Step 4 — verify shapes and element count.** Input `b*C*h*w` elements; output `b*c*(h*r)*(w*r)` = `b*(C/(r*r))*h*r*w*r` = `b*C*h*w`. Counts match, and the printed output shape should be `(b, c, h*r, w*r)`.

In [ ]:
def depth_to_space(x: Tensor, r: int) -> Tensor:
    return rearrange(x, 'b (c r1 r2) h w -> b c (h r1) (w r2)', r1=r, r2=r)


np.random.seed(0)
t.manual_seed(0)
x = t.randn(2, 3 * 2 * 2, 4, 5)  # b=2, C=12 (c=3, r=2), h=4, w=5
out = depth_to_space(x, r=2)
print('input shape :', tuple(x.shape))
print('output shape:', tuple(out.shape), '(expect (2, 3, 8, 10))')
print('element count preserved?', x.numel() == out.numel())